# Huấn luyện YOLOv8 trên Google Colab (MIỄN PHÍ GPU)

Đây là file Notebook hoàn chỉnh. Bạn chỉ cần upload file này lên [Google Colab](https://colab.research.google.com/), sau đó chạy lần lượt từng ô code từ trên xuống dưới.

**Lưu ý:** 
1. Nén thư mục `dataset/` thành file `dataset.zip` và tải lên thư mục gốc của Google Drive.
2. Bật GPU cho Colab: Vào `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> Chọn **T4 GPU**.

In [ ]:
# Ô 1: Cài đặt thư viện & Kiểm tra GPU
!pip install ultralytics
import torch
print(f"Sẵn sàng huấn luyện trên: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### **Ô 2: Kết nối với Google Drive**
Một cửa sổ sẽ hiện ra yêu cầu bạn cấp quyền truy cập Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **Ô 3: Giải nén Dataset**
Đảm bảo file `dataset.zip` nằm ở thư mục gốc của Google Drive.

In [ ]:
!unzip /content/drive/MyDrive/dataset.zip -d /content/

### **Ô 3.5 (Tùy chọn): Kiểm tra lại đường dẫn**
Nếu bạn gặp lỗi `FileNotFoundError` ở bước sau, hãy chạy ô này để kiểm tra xem dataset đã được giải nén đúng cấu trúc thư mục chưa.

In [ ]:
# Chạy lệnh này để xem các file trong thư mục /content
!ls /content/

# Bạn phải thấy thư mục "dataset" được liệt kê ở trên
# Chạy tiếp lệnh này để xem bên trong thư mục dataset
!ls /content/dataset/

# Bạn phải thấy file "dataset.yaml" và các thư mục con

### **Ô 4: Bắt đầu Huấn luyện**
Quá trình này sẽ mất khoảng 15-30 phút tùy vào số lượng ảnh.

In [ ]:
from ultralytics import YOLO

# 1. Khởi tạo model nano (Nhẹ & Nhanh)
model = YOLO('yolov8n.pt')

# 2. Huấn luyện với đa luồng (workers=8)
# Lưu ý: file dataset.yaml cần có path: /content/dataset/
model.train(data='/content/dataset/dataset.yaml', epochs=50, imgsz=400, device=0, workers=8, batch=16)

### **Ô 5: Xuất sang ONNX & Tải về**
Sau khi chạy ô này, trình duyệt sẽ tự động tải file `best.onnx` về máy tính của bạn.

In [ ]:
import os

# Đường dẫn tới model tốt nhất sau khi train
# Colab có thể tạo ra các thư mục train, train2, train3... nên ta cần tìm thư mục mới nhất
latest_run = sorted([d for d in os.listdir('/content/runs/detect') if d.startswith('train')])[-1]
best_model_path = f'/content/runs/detect/{latest_run}/weights/best.pt'
onnx_output_path = f'/content/runs/detect/{latest_run}/weights/best.onnx'

if os.path.exists(best_model_path):
    # Load model đã train và xuất sang ONNX
    trained_model = YOLO(best_model_path)
    trained_model.export(format='onnx', imgsz=400)
    
    # Download file về máy tính
    from google.colab import files
    files.download(onnx_output_path)
    print(f"Đang tải file {onnx_output_path} về máy của bạn...")
else:
    print(f"Không tìm thấy model tại {best_model_path}. Hãy kiểm tra lại thư mục 'runs'.")